# Title: RAGCHECKER: A Fine-grained Framework for Diagnosing Retrieval-Augmented Generation

#### Members' Names: Amadike Lilian, Jeff Ogakwu, Philemon David

####  Emails: camadike@torontomu.ca, david.philemon@torontomu.ca, jeff.ogakwu@torontomu.ca

# Introduction:

#### **Problem Description:**

Retrieval-Augmented Generation (RAG) systems improve large language models by retrieving external documents and using them to generate answers. Although RAG can produce more factual and context-aware responses than standalone language models, evaluating its performance is difficult. Since a RAG system has both a retriever and a generator, errors in the final response may come from retrieving irrelevant information, missing useful information, or generating unsupported claims from the retrieved context. The RAGChecker paper identifies this as a major challenge in RAG evaluation, especially for long-form responses where coarse metrics do not capture fine-grained errors.


#### **Context of the Problem:**

This problem is important because RAG systems are now used in applications where factual accuracy and reliability matter, such as question answering and knowledge-based assistance. If evaluation is weak, it becomes hard to understand whether poor performance is caused by retrieval failure or generation failure. A more detailed evaluation is therefore necessary for diagnosing system behavior and improving the overall quality of responses.


#### **Limitation About other Approaches:**

Earlier approaches often rely on metrics such as BLEU, ROUGE, recall@k, or other coarse response-level measures. These approaches do not clearly separate retriever errors from generator errors, and they are not detailed enough to assess claim-level correctness in long answers.


#### **Solution:**

To overcome these challenges, we introduce RAGCHECKER, an innovative evaluation framework designed for detailed analysis of both retrieval and generation processes. RAGCHECKER is based onclaim-level entailment checking which involves operations of extracting claims from the response and ground truth answer and checking them against other texts. This approach enables fine-grained evaluation instead of response-level assessment. RAGCHECKER processes the user query, retrieved context, response, and ground truth answer, producing a suite of metrics:

- **Overall Metrics** to provide a holistic view of the system performance, assessing the overall quality of the generated responses.
-  **Diagnostic Retriever Metrics** to evaluate the effectiveness of the retriever, identifying its strengths and weaknesses in finding relevant information from the knowledge base.
- **Diagnostic Generator Metrics** to assess the performance of the generator, diagnosing how well the generator utilizes the retrieved context, handles noisy information, and generates accurate and faithful responses.

# Background


| Reference |Explanation |  Dataset/Input |Weakness
| --- | --- | --- | --- |
| Lewis et al. (RAG, 2020) [1]| They introduced the original RAG framework combining dense retrieval with sequence generation for knowledge-intensive tasks | Open-domain QA datasets (e.g., Natural Questions, Wikipedia corpus) | Focuses on model performance, not detailed evaluation of RAG components
| Es et al. (RAGAS) [2]| They evaluated RAG systems using metrics such as answer relevance, context relevance, and groundedness via LLM scoring | QA datasets with generated responses | Coarse-grained; cannot separate retriever vs generator errors
| Ferrara et al. (TruLens) [3] | They proposed the RAG Triad (context relevance, groundedness, answer relevance) for evaluating RAG pipelines| Query, retrieved context, and generated responses | Relies on LLM judgments; lacks fine-grained diagnostic capability
| Chen et al. (RGB Benchmark) [4] | They evaluated generator robustness in RAG systems across tasks like noise handling and information integration| Manually constructed QA datasets with perturbed contexts | Focuses only on generator; does not evaluate retrieval component
| Saad-Falcon et al. (ARES) [5] | They automated a framework for evaluating RAG systems using learned scoring models| QA datasets with generated outputs | Limited interpretability; focuses on overall scores rather than detailed error analysis
| Ru et al. (RAGChecker, 2024) [6] | They introduced claim-level evaluation to assess both retriever and generator using metrics like claim recall, context precision, and faithfulness| ClapNQ, RobustQA, LoTTE, KIWI, NovelQA | Computationally expensive; requires claim extraction and entailment models


# Methodology

This project implements a Retrieval-Augmented Generation (RAG) pipeline combined with a fine-grained evaluation framework inspired by RAGChecker. The methodology follows a structured sequence of steps, beginning with dataset preparation and ending with comparative performance analysis across multiple system configurations. The entire process can be understood as a sequence of structured steps:

![RAGChecker Methodology](https://drive.google.com/uc?export=view&id=1kViD1G5kwDRaGZFAK7cG0llxjyvhdbB4)

**Figure:** RAGChecker methodology pipeline showing dataset loading, parsing, corpus preparation, retrieval using FAISS and BM25, response generation, system construction, evaluation using claim-level metrics (precision, recall, claim recall, context precision, faithfulness, hallucination), and final analysis.
#

###**Step 1: Dataset Loading**

Two question-answering datasets are used for evaluation: NovelQA and ClapNQ. NovelQA is loaded from locally cached JSON files, while ClapNQ is accessed through the Hugging Face datasets library. In addition, the LoTTE science corpus is loaded using ir_datasets and serves as the external knowledge base for retrieval. This setup ensures a clear separation between evaluation queries and the document corpus used for retrieval.

#

###**Step 2: Dataset Parsing and Structuring**

The raw datasets are transformed into a consistent format. For NovelQA, each sample is parsed to extract the question, ground-truth answer, and supporting evidence. For ClapNQ, the query and reference answer are extracted and stored in the same structure. This standardization ensures compatibility across datasets and simplifies downstream processing.

#

###**Step 3: Retrieval Corpus Preparation**

The LoTTE corpus is processed into a structured document collection. Each document is assigned an identifier and stored as text, forming the retrieval corpus. This corpus is then prepared for efficient search during the retrieval stage.

#

###**Step 4: Retrieval Using FAISS and BM25**

Two retrieval strategies are implemented to obtain relevant context for each query:

Dense Retrieval (FAISS): Documents are embedded using a SentenceTransformer model (all-MiniLM-L6-v2) and indexed using FAISS. Queries are embedded into the same vector space, and the top-
𝑘
k most similar documents are retrieved based on semantic similarity.
Sparse Retrieval (BM25): Documents are tokenized and indexed using BM25, which retrieves documents based on lexical matching between query terms and document content.

These approaches enable comparison between semantic and keyword-based retrieval within the same pipeline.

#

###**Step 5: Response Generation**

Retrieved documents are passed to a generator to produce answers. Two generation strategies are used:

- Extractive baseline: returns answers directly from retrieved content
- GPT-based generator: uses a large language model to generate responses conditioned on the retrieved context

This enables comparison between non-generative and generative approaches.

#

###**Step 6: Construction and Formatting of RAG Systems**

Four RAG system configurations are constructed by combining retrievers and generators:

1.) FAISS + Extractive

2.) BM25 + Extractive

3.) FAISS + GPT

4.) BM25 + GPT

Each system follows the same pipeline but differs in how context is retrieved and how responses are generated. The outputs of each system, including generated responses and retrieved documents, are then converted into a standardized format compatible with the RAGChecker evaluation framework. This step ensures seamless integration between the implemented pipeline and the evaluation process.

#

###**Step 7: Evaluation Using RAGChecker**

Each system is evaluated on both NovelQA and ClapNQ datasets using a fine-grained, claim-level evaluation approach. The evaluation decomposes responses into individual claims and compares them against ground-truth answers and retrieved documents.

The following metrics are computed:

- Precision
- Recall
- Claim Recall
- Context Precision
- Faithfulness
- Hallucination

![RAGChecker Evaluation Metrics](https://drive.google.com/uc?export=view&id=1vjuKLckyxvGM486sqmwGZ1ek2ToE73e9)

**Figure:** RAGChecker evaluation metrics used in this project. The metrics include overall performance measures (precision and recall), retriever metrics (claim recall and context precision), and generator metrics (faithfulness and hallucination), providing a fine-grained analysis of RAG system behavior.

These metrics provide detailed insights into both overall system performance and the behavior of individual components.

#

###**Step 8: Aggregation of Results**

Evaluation results from NovelQA and ClapNQ are combined by averaging the metric values across datasets. This provides a more robust and generalized comparison of system performance.

#

###**Step 9: Analysis and Comparison**

The final step involves analyzing the performance of all four RAG systems.

![RAG System Analysis and Comparison](https://drive.google.com/uc?export=view&id=1FrGbhr894uGLqORpxu8vDKiV0-mch-GU)

**Figure:** Comparison of the four RAG systems showing that FAISS improves recall, GPT improves precision, and BM25 reduces hallucination, highlighting trade-offs between coverage and reliability.

The comparison focuses on:

- Differences between FAISS and BM25 retrieval
- Improvements introduced by GPT-based generation
- Trade-offs between faithfulness and hallucination
- The relationship between retrieval quality and response accuracy

This analysis provides insights into how different design choices impact the effectiveness of RAG systems.



# Implementation



###**1. Setup (clone + install)**

In [ ]:
# Uncomment codes below to navigate to content folder, clone repo and install necessary dependencies

#%cd /content

#!git clone https://github.com/amazon-science/RAGChecker.git || echo "Repo exists"

#%cd /content/RAGChecker
#!pip install -e .

#%cd /content

# Let pip resolve compatible versions automatically
#!pip install -q sentence-transformers==2.7.0 peft==0.11.1 faiss-cpu rank-bm25 datasets

In [ ]:
# Inspect RAGChecker directory /RAGChecker
!ls /content/RAGChecker

### **2. Data Loading**

### We will download LoTTE using ir_datasets

In [ ]:
# Uncomment the code below to load
#!pip install ir_datasets

Load LoTTE documents and querys (Science domain) - LoTTE will be as retrieval corpus (documents) not for query → answer evaluation

In [ ]:
import ir_datasets

dataset = ir_datasets.load("lotte/science/dev/search")

Inspect Data (Queries and Documents) in the LoTTE

In [ ]:
queries = list(dataset.queries_iter())[:20]
docs = list(dataset.docs_iter())[:2000]  # Limit size

print("Sample query:", queries[0])
print("Sample doc:", docs[0])

Login to Hugging Face and enter access token need for NovelQA and ClapNQ datasets

In [ ]:
from huggingface_hub import login

login()

### Download NovelQA (Question + Answer) - dataset 1

**Note**: We would get a DatasetGenerationCastError here but it is expected and harmless because NovelQA has inconsistent JSON schema. But overall:
- Dataset will download successfully
- JSON files (B70.json, B66.json, etc.) will be in cache

In [ ]:
from datasets import load_dataset

# Uncomment the code below to download
#_ = load_dataset("NovelQA/NovelQA", split="train[:1]")

Find where NovelQA data was stored so that we can use it

In [ ]:
import json
from pathlib import Path

search_paths = [
    Path("/content"),
    Path("/root/.cache/huggingface")
]

all_json = []

# Extracts valid Question + Answer queries
for base in search_paths:
    if base.exists():
        all_json.extend(list(base.rglob("B*.json")))

print("Found:", len(all_json))

### Load and Parse NovelQA dataset
We also extract evidence from NovelQA data, will be used to compare retrieved docs vs evidence & analyze retrieval quality.
The evaluation was conducted on a subset of NovelQA queries due to data accessibility constraints and the computational cost of LLM-based evaluation. Despite the limited size, the dataset provides sufficient diversity to analyze comparative trends across different RAG configurations

In [ ]:
novelqa_samples = []

for file in all_json:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for q_id, q_data in data.items():
        question = q_data.get("Question", "").strip()
        answer = q_data.get("Answer", "").strip()

        evidence_texts = []
        for ev in q_data.get("Evidences", {}).values():
            evidence_texts.append(ev.get("Evidence", "").strip())

        evidence = " ".join([e for e in evidence_texts if e])

        if question and answer:
            novelqa_samples.append({
                "query_id": q_id,
                "query": question,
                "gt_answer": answer,
                "evidence": evidence
            })

print(novelqa_samples[:2])

### Load ClapNQ data from from PrimeQA (Question + Answer) - Dataset 2

In [ ]:
from datasets import load_dataset

ds = load_dataset("PrimeQA/clapnq")
len(ds["train"]) # Size of ClapNQ Dataset

In [ ]:
# Inspect ClapNQ data to confirm structure
print(ds)
print(type(ds["train"]))
print(ds["train"][0])

Structure the data loaded from ClapNQ properly, in the way the RAG systems expects

In [ ]:
subset = ds["train"].select(range(50))

clapnq_samples = []

for i, item in enumerate(subset):

    question = item["input"].strip()

    answers = item.get("output", [])

    if answers and isinstance(answers, list):
        answer = answers[0].get("answer", "").strip()
    else:
        answer = ""

    if question and answer:
        clapnq_samples.append({
            "query_id": str(i),
            "query": question,
            "gt_answer": answer
        })

print("Total CLAPNQ samples:", len(clapnq_samples))

Prepare LoTTE data for FAISS embedding

In [ ]:
lotte_queries = [
    {"query_id": q.query_id, "query": q.text}
    for q in queries
]

lotte_corpus = [
    {"doc_id": d.doc_id, "text": d.text}
    for d in docs
]

print(lotte_queries[:2])
print(lotte_corpus[:2])

### **3. Retrieval**

We employ a lightweight dense retriever (FAISS) instead of more advanced models such as E5-Mistral used in the paper due to computational constraints, and Sparse retriever (BM25) as used in the research paper.  While stronger embedding models may improve retrieval performance, our focus is on comparing retrieval strategies and generation methods rather than optimizing the retriever itself. Therefore, the chosen setup is sufficient to demonstrate the relative strengths of sparse and dense retrieval within the RAG framework.

### Build FAISS Retriever

FAISS retrieves documents using semantic similarity by comparing embeddings instead of exact keywords.

In [ ]:
# Install + Imports

!pip install -q sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

Create Embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [d.text for d in docs]   # from LoTTE
doc_ids = [d.doc_id for d in docs]

embeddings = model.encode(texts, normalize_embeddings=True)
embeddings = np.array(embeddings).astype("float32")

Build FAISS index

In [ ]:
# Facebook AI Similarity Search
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index size:", index.ntotal)

Retrival function (FAISS)

In [ ]:
def retrieve_faiss(query, top_k=5):
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "doc_id": doc_ids[idx],
            "text": texts[idx],
            "score": float(scores[0][i])
        })
    return results

### Build BM25 Retriever
BM25 (Best Matching 25) is a ranking algorithm that retrieves documents based on keyword matching, using term frequency and inverse document frequency to score relevance.

In [ ]:
# Best Matching 25
# Setup BM25
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(t) for t in texts]

bm25 = BM25Okapi(tokenized_corpus)

Retrival function (BM25)

In [ ]:
def retrieve_bm25(query, top_k=5):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "doc_id": doc_ids[idx],
            "text": texts[idx],
            "score": float(scores[idx])
        })
    return results

### Test both FAISS AND BM25 Retrievers

FAISS uses semantic similarity (embeddings), while BM25 uses keyword-based matching (term frequency scoring).
#
In simple terms
- Keyword matching → looks at words
- Semantic similarity → looks at meaning

In [ ]:
test_query = "is sudan iv hydrophobic or hydrophilic?"

print("FAISS RESULTS:")
for r in retrieve_faiss(test_query, top_k=3):
    print("-", r["text"][:120])

print("\nBM25 RESULTS:")
for r in retrieve_bm25(test_query, top_k=3):
    print("-", r["text"][:120])

Key Insights from testing the two Retrievers

- Neither system retrieves the exact answer
- But both retrieve partially useful context

Why Results Aren’t Perfect

Because:

LoTTE = general corpus (StackExchange)

The query = specific chemistry question

So no guarantee exact answer exists
retrieval is approximate

**This tells us that Retrieval quality varies across methods, and relevant but incomplete context can still be retrieved even when exact answers are absent.**

### **4. RAG Pipeline**

### Now to build our 4 RAG Systems, we define:
Generator A = extractive baseline (No LLM)

Generator B = OpenAI GPT (With LLM)

In [ ]:
from openai import OpenAI
import os
from getpass import getpass

# Run this once per session
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

def generate_extractive(query, docs, max_chars=400):
    context = " ".join([d["text"] for d in docs])
    return f"Answer: Based on the retrieved information, {context[:max_chars]}"

def generate_gpt(query, docs, model_name="gpt-4o-mini"):
    context = "\n\n".join([f"[Doc {i+1}] {d['text']}" for i, d in enumerate(docs)])

    prompt = f"""
You are answering a question using only the retrieved context below.

Question:
{query}

Retrieved context:
{context}

Write a concise answer grounded only in the retrieved context.
If the context is insufficient, say so clearly.
""".strip()

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

### Build full RAG Pipeline

run_rag() function that supports both retrievers and both generators (baseline/ No LLM and GPT/ With specified LLM)

In [ ]:
def run_rag(query, retriever="faiss", generator="extractive", top_k=5):
    if retriever == "faiss":
        docs = retrieve_faiss(query, top_k=top_k)
    elif retriever == "bm25":
        docs = retrieve_bm25(query, top_k=top_k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if generator == "extractive":
        response = generate_extractive(query, docs)
    elif generator == "gpt":
        response = generate_gpt(query, docs)
    else:
        raise ValueError(f"Unknown generator: {generator}")

    return response, docs

We defined the four RAG systems explicitly

| Retriever | Generator                |
| --------- | ------------------------ |
| FAISS     | Extractive (No LLM) |
| BM25      | Extractive (No LLM) |
| FAISS     | GPT                      |
| BM25      | GPT                      |


In [ ]:
systems = [
    {"name": "FAISS_Extractive", "retriever": "faiss", "generator": "extractive", "top_k": 5},
    {"name": "BM25_Extractive",  "retriever": "bm25",  "generator": "extractive", "top_k": 5},
    {"name": "FAISS_GPT",        "retriever": "faiss", "generator": "gpt",        "top_k": 5},
    {"name": "BM25_GPT",         "retriever": "bm25",  "generator": "gpt",        "top_k": 5},
]

### Build function to run the 4 RAG Systems ( code extracted from the github repo)
Used a smaller limit because GPT evaluation plus RAGChecker claim extraction has high computational cost

In [ ]:
def build_results_for_system(samples, retriever, generator, top_k=5, limit=5):
    results = []

    for sample in samples[:limit]:
        response, docs = run_rag(
            sample["query"],
            retriever=retriever,
            generator=generator,
            top_k=top_k
        )

        results.append({
            "query_id": sample["query_id"],
            "query": sample["query"],
            "ground_truth": sample["gt_answer"],
            "response": response,
            "documents": docs,
            "evidence": sample.get("evidence", "")
        })

    return results

### **5. Monkey Patch litellm**

### Set RAGChecker to use OpenAI instead of falling back to Bedrock

We do this because Bedrock will fail since we have no AWS credentials, so we used OpenAI which we have instead.

In [ ]:
import litellm

original_completion = litellm.completion

def force_openai_model(*args, **kwargs):
    # Only override if model is missing or bedrock
    if "model" not in kwargs or "bedrock" in str(kwargs.get("model", "")):
        kwargs["model"] = "gpt-4o-mini"

    return original_completion(*args, **kwargs)

litellm.completion = force_openai_model

Load and view their container.py to identify the modules needed to wrap out results into RAGResult / RAGResults format, by uncommenting the code below

In [ ]:
#!sed -n '1,200p' /content/RAGChecker/ragchecker/container.py

Add helper function extracted from code repo, to wrap our results into RAGResult / RAGResults

In [ ]:
# Add RAGChecker to python path to make sure import doesn't fail
import sys
sys.path.append("/content/RAGChecker")

In [ ]:
from ragchecker.container import RAGResults, RAGResult
from types import SimpleNamespace

def to_ragchecker_input(results):
    ragchecker_results = []

    for item in results:

        # Convert dict → object with .text
        retrieved_context = [
            SimpleNamespace(doc_id=d["doc_id"], text=d["text"])
            for d in item["documents"]
        ]

        ragchecker_results.append(
            RAGResult(
                query_id=str(item["query_id"]),
                query=item["query"],
                gt_answer=item["ground_truth"],
                response=item["response"],
                retrieved_context=retrieved_context
            )
        )

    return RAGResults(results=ragchecker_results)

### **6. Evaluation**
We will use the RAGChecker to evaluate the 4 RAG system on both datasets (NovelQA & ClapNQ)

Load and view the evaluator.py from the code repo by uncommenting the code below

This shows:

- How evaluation is actually done

- What functions/classes are used

In [ ]:
#!sed -n '1,200p' /content/RAGChecker/ragchecker/evaluator.py

### Implement RAGChecker to evaluate each of the 4 RAG systems and get the fine-grained metrics

**NovelQA Dataset Evaluation Using RAGChecker (code adapted from their github repo)**

In [ ]:
from ragchecker import RAGChecker
import pandas as pd

checker = RAGChecker()

all_eval_rows = []
all_raw_evals = {}

for sys in systems:
    print(f"Running {sys['name']} ...")

    sys_results = build_results_for_system(
        samples=novelqa_samples,
        retriever=sys["retriever"],
        generator=sys["generator"],
        top_k=sys["top_k"],
        limit=20
    )

    rag_input = to_ragchecker_input(sys_results)
    evaluation = checker.evaluate(rag_input)
    all_raw_evals[sys["name"]] = evaluation

    # flatten helper
    if isinstance(evaluation, dict):
        row = {"System": sys["name"], **evaluation}
    else:
        # fallback if evaluation is an object
        row = {"System": sys["name"]}
        for attr in dir(evaluation):
            if not attr.startswith("_"):
                val = getattr(evaluation, attr)
                if isinstance(val, (int, float, str)):
                    row[attr] = val

    all_eval_rows.append(row)

results_df = pd.DataFrame(all_eval_rows)
results_df

Clean version of the Metrics table for NovelQA

In [ ]:
import pandas as pd

clean_rows = []

for system, eval_obj in all_raw_evals.items():

    overall = eval_obj["overall_metrics"]
    retriever = eval_obj["retriever_metrics"]
    generator = eval_obj["generator_metrics"]

    clean_rows.append({
        "System": system,
        "Precision": round(float(overall["precision"]), 2),
        "Recall": round(float(overall["recall"]), 2),
        "Claim Recall": round(float(retriever["claim_recall"]), 2),
        "Context Precision": round(float(retriever["context_precision"]), 2),
        "Faithfulness": round(float(generator["faithfulness"]), 2),
        "Hallucination": round(float(generator["hallucination"]), 2)
    })

df_novelqa = pd.DataFrame(clean_rows)
df_novelqa

![NovelQA Metrics Table](https://drive.google.com/uc?export=view&id=1knYKRm-ofRiJLFurvKB9edUAkjWFdSLc)


In addition to the metrics used in our experiments, the RAGChecker framework defines additional generator-level metrics such as context utilization, noise sensitivity, and self-knowledge.

Why We didn’t use those

Because:

- RAGChecker doesn’t expose all metrics easily
- They require deeper config
- They require more LLM calls → more cost (large dataset size)

**ClapNQ Dataset Evaluation Using RAGChecker**

In [ ]:
all_raw_evals_clapnq = {}

for sys in systems:

    print(f"Running {sys['name']} on CLAPNQ...")

    sys_results = []

    for sample in clapnq_samples:
        response, docs = run_rag(
            sample["query"],
            retriever=sys["retriever"],
            generator=sys["generator"]
        )

        sys_results.append({
            "query_id": sample["query_id"],
            "query": sample["query"],
            "ground_truth": sample["gt_answer"],
            "response": response,
            "documents": docs
        })

    rag_input = to_ragchecker_input(sys_results)

    evaluation = checker.evaluate(rag_input)

    all_raw_evals_clapnq[sys["name"]] = evaluation

Clean version of the Metrics table for ClapNQ

In [ ]:
clean_rows_clapnq = []

for system, eval_obj in all_raw_evals_clapnq.items():

    overall = eval_obj["overall_metrics"]
    retriever = eval_obj["retriever_metrics"]
    generator = eval_obj["generator_metrics"]

    clean_rows_clapnq.append({
        "System": system,
        "Precision": round(float(overall["precision"]), 2),
        "Recall": round(float(overall["recall"]), 2),
        "Claim Recall": round(float(retriever["claim_recall"]), 2),
        "Context Precision": round(float(retriever["context_precision"]), 2),
        "Faithfulness": round(float(generator["faithfulness"]), 2),
        "Hallucination": round(float(generator["hallucination"]), 2)
    })

df_clapnq = pd.DataFrame(clean_rows_clapnq)
df_clapnq

![ClapNQ Metrics Table](https://drive.google.com/uc?export=view&id=1VhVly__G-WQ6rRE_OaPiNccE_tmQICuo)

Average Performace across both the NovelQA and ClapNQ dataset

In [ ]:
# Set index to System for both
df_nq = df_novelqa.set_index("System")
df_cq = df_clapnq.set_index("System")

# Take average across datasets
df_avg = (df_nq + df_cq) / 2

df_avg

df_paper = df_avg.reset_index()

df_paper = df_paper[[
    "System",
    "Precision",
    "Recall",
    "Claim Recall",
    "Context Precision",
    "Faithfulness",
    "Hallucination"
]]

df_paper.columns = pd.MultiIndex.from_tuples([
    ("RAG systems", ""),
    ("Overall", "Precision"),
    ("Overall", "Recall"),
    ("Retriever", "Claim Recall"),
    ("Retriever", "Context Precision"),
    ("Generator", "Faithfulness"),
    ("Generator", "Hallucination"),
])

df_paper = df_paper.round(2)

df_paper

![Averaged Metrics Table](https://drive.google.com/uc?export=view&id=1gRGYPLD2UGNf08IHzkBXznty8vgQh0Bv)

### **7. Analysis**

### Analysis of Averaged RAG System Performance

**1. Overall Performance (Precision & Recall)**

Across both datasets, GPT-based systems consistently outperform extractive baselines in terms of precision. FAISS_GPT achieves the highest average precision (29.8), followed by BM25_GPT (26.9), while extractive systems remain significantly lower (≤2.7). This confirms that generative models substantially improve answer correctness by synthesizing information from retrieved context rather than relying on direct extraction.

In contrast, recall values remain relatively moderate across all systems. FAISS-based systems achieve the highest recall (9.0), while BM25-based systems show slightly lower values (7.5). This aligns with the RAGChecker framework, where recall is computed at the claim level, making it inherently strict. Even partially correct answers may not fully match ground-truth claims, resulting in lower recall scores.

Overall, these results highlight that generation primarily improves precision, while recall remains constrained by both retrieval quality and strict evaluation criteria.

**2. Retriever Performance (Claim Recall & Context Precision)**

Retriever performance reveals a consistent pattern across datasets:

FAISS-based systems achieve higher claim recall (9.9), indicating stronger semantic retrieval capabilities. This suggests that dense retrieval methods are more effective at capturing relevant information, especially when queries require semantic understanding.

BM25-based systems, however, exhibit slightly lower claim recall (6.7), reflecting their reliance on lexical matching. Despite this limitation, BM25 retrieval remains competitive due to its ability to return more focused and less noisy results.

Context precision remains moderate across all systems (8–10 range), indicating that both retrievers retrieve a mixture of relevant and irrelevant content. This supports the observation in the RAGChecker paper that retrieval quality alone is insufficient, and that downstream generation must handle noisy context effectively.

**3. Generator Performance (Faithfulness & Hallucination)**

Generator performance highlights important trade-offs between correctness and reliability.

FAISS_Extractive achieves the highest faithfulness (44.2), followed by FAISS_GPT (33.0). This initially appears surprising, but can be explained by the nature of extractive methods, which directly copy retrieved text. While this leads to high faithfulness, it does not necessarily result in correct or complete answers, as reflected by their very low precision.

GPT-based systems, while slightly lower in faithfulness, provide significantly better overall answer quality. This demonstrates that faithfulness alone is not sufficient, and must be interpreted alongside precision and recall.

In terms of hallucination:

BM25_Extractive achieves the lowest hallucination (5.3), followed by BM25_GPT (19.7).
FAISS_GPT exhibits the highest hallucination (24.0), reflecting the impact of noisier semantic retrieval.

These findings reinforce the RAGChecker paper’s emphasis that hallucination and faithfulness are critical complementary metrics, capturing both correctness and grounding of generated responses.

**4. Trade-offs Between FAISS and BM25**

A clear and consistent trade-off emerges between dense (FAISS) and sparse (BM25) retrieval methods:

FAISS (dense retrieval) provides better semantic coverage, leading to higher claim recall and recall overall. However, it introduces more noise into the retrieved context, which increases hallucination during generation.
BM25 (sparse retrieval) produces more precise and focused context, resulting in lower hallucination and more stable outputs. However, it may miss semantically relevant information, leading to lower recall.

This trade-off highlights a key insight:

Better retrieval coverage (FAISS) vs cleaner retrieval precision (BM25)

**5. Best Performing System**

When averaging across both datasets, FAISS_GPT emerges as the strongest overall system in terms of precision and recall, benefiting from its ability to combine semantic retrieval with generative reasoning.

However, BM25_GPT remains the most balanced system, achieving:

competitive precision
lower hallucination
stable performance across datasets

This suggests that while FAISS_GPT excels in coverage, BM25_GPT provides more reliable and controlled outputs.

**6. Key Insight: Generalization Across Datasets**

Averaging results across NovelQA and CLAPNQ reveals an important conclusion:

RAG system performance is highly dependent on dataset characteristics
On reasoning-heavy datasets (NovelQA), generation plays a dominant role
On factual datasets (CLAPNQ), retrieval quality becomes more important

By averaging across datasets, we capture generalization ability, showing that:

FAISS-based systems generalize better in recall
BM25-based systems generalize better in reliability (lower hallucination)
GPT-based generation consistently improves precision across both settings

# Conclusion and Future Direction

In this project, we implemented and evaluated different Retrieval-Augmented Generation (RAG) systems to better understand how retrieval and generation interact. One of the main things we observed is that the performance of a RAG system does not depend on just one component. Instead, both the retriever and the generator play important roles, and their interaction directly affects the final output.

From the results, FAISS-based retrieval performed better in terms of capturing relevant information compared to BM25, mainly because it uses semantic similarity instead of exact keyword matching. However, this also introduced more noise in some cases. On the generation side, the GPT-based models clearly improved precision and produced more complete answers compared to the extractive baseline. At the same time, they also introduced hallucination, showing that better generation does not always mean more reliable answers.

Another important takeaway from this project is the usefulness of fine-grained evaluation. Using metrics like claim recall, faithfulness, and hallucination made it much easier to understand where errors were coming from. Instead of just looking at overall performance, it became possible to see whether issues were caused by poor retrieval or by the generator not using the context properly.

There are still some limitations in this work. The evaluation was done on a limited set of datasets, and only a subset of the available RAGChecker metrics was used. Also, the generator was treated as a black box, so there was no direct control over how it reasons or uses the retrieved information.

For future work, several improvements can be explored. More advanced retrieval models could be used to reduce noise while maintaining high recall. Better prompting or controlled generation techniques could help reduce hallucination and improve faithfulness. It would also be useful to test the system on more datasets and include additional evaluation metrics for a more complete analysis. Finally, adding a filtering or reranking step before generation could help improve the quality of retrieved context and lead to more reliable responses.

Overall, this project helped build a deeper understanding of how RAG systems work in practice and showed that careful evaluation is necessary to properly analyze and improve their performance.

# References:

[1]: D. Ru, L. Qiu, X. Hu, T. Zhang, P. Shi, S. Chang, C. Jiayang, C. Wang, S. Sun, H. Li, Z. Zhang, B. Wang, J. Jiang, T. He, Z. Wang, P. Liu, Y. Zhang, and Z. Zhang, “RAGChecker: A Fine-grained Framework for Diagnosing Retrieval-Augmented Generation,” arXiv preprint arXiv:2408.08067, 2024.

[2]: P. Lewis, E. Perez, A. Piktus, F. Petroni, V. Karpukhin, N. Goyal, H. Küttler, M. Lewis, W. Yih, T. Rocktäschel, S. Riedel, and D. Kiela, “Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks,” in Proc. Adv. Neural Inf. Process. Syst. (NeurIPS), 2020.

[3]: E. Es, J. James, L. Espinosa Anke, and S. Schockaert, “RAGAS: Automated Evaluation of Retrieval Augmented Generation,” in Proc. Conf. Empirical Methods in Natural Language Processing (EMNLP), 2023.

[4]: F. Ferrara, A. Pandey, and H. Chase, “TruLens: A Framework for Evaluating Large Language Model Applications,” arXiv preprint arXiv:2307.13970, 2023.

[5]: M. Saad-Falcon, D. Xu, and C. Callison-Burch, “ARES: An Automated Evaluation Framework for Retrieval-Augmented Generation Systems,” in Proc. Annual Meeting of the Association for Computational Linguistics (ACL), 2024.

[6]: J. Chen, H. Lin, X. Han, and L. Sun, “Benchmarking Large Language Models in Retrieval-Augmented Generation,” arXiv preprint arXiv:2309.01431, 2023.